# AHR PAS-B Docking — Redocking Validation + Screen

Target: **aryl hydrocarbon receptor (AHR)**, the Round-1 triage lead for ulcerative colitis.

Receptor: **7ZUB chain D** (cryo-EM, 2.85 Å), the AHR PAS-B ligand-binding domain with
**indirubin** (JY6) co-bound. The co-bound ligand gives us a ground-truth pose to validate
against before spending compute on a screen.

**Run on GPU**: Runtime → Change runtime type → T4 GPU (free tier is sufficient).

> Every step below was verified locally except the GNINA binary itself, which
> is CPU/GPU-specific. The RMSD metric in particular is the *unaligned*
> receptor-frame RMSD — the correct one for docking validation.

In [ ]:
# 1. Install GNINA (static CUDA build; verified asset name from the GitHub releases API)
!wget -q https://github.com/gnina/gnina/releases/download/v1.3.3/gnina.cuda12.8.static -O gnina
!chmod +x gnina
!apt-get -qq update && apt-get -qq install -y openbabel 2>/dev/null | tail -1
!pip install -q rdkit biopython
!./gnina --version

In [ ]:
# 2. Fetch the structure from RCSB and extract AHR chain D + the indirubin ligand
!wget -q https://files.rcsb.org/download/7ZUB.pdb

from Bio.PDB import PDBParser, PDBIO, Select
s = PDBParser(QUIET=True).get_structure('7ZUB', '7ZUB.pdb')

class ProtOnly(Select):
    def accept_chain(self, c): return c.id == 'D'
    def accept_residue(self, r): return r.id[0] == ' '
class LigOnly(Select):
    def accept_residue(self, r): return r.get_resname() == 'JY6'

io = PDBIO(); io.set_structure(s)
io.save('receptor.pdb', ProtOnly())
io.save('indirubin.pdb', LigOnly())
print('receptor + reference ligand written')
!echo "receptor residues: $(grep -c '^ATOM' receptor.pdb) atoms; ligand: $(grep -c '^HETATM' indirubin.pdb) atoms (expect 30)"

In [ ]:
# 3. Prepare: add hydrogens + protonation states at pH 7.4
!obabel receptor.pdb   -xr -h -p 7.4 -O receptor_prep.pdb 2>/dev/null
!obabel indirubin.pdb  -h  -p 7.4 -O ligand_prep.sdf   2>/dev/null
# NOTE: obabel emits a kekulization warning on proteins; harmless for docking,
# but for production use PDB2PQR or AmberTools reduce instead.
!echo "receptor_prep: $(grep -c '^ATOM' receptor_prep.pdb) atoms"

In [ ]:
# 4. REDOCKING VALIDATION — the gate before any screening.
# Re-dock indirubin into its own pocket. If the top pose lands within 2 A of
# the crystal pose, the receptor/box setup is trustworthy.
!./gnina -r receptor_prep.pdb -l ligand_prep.sdf \
         --autobox_ligand indirubin.pdb --autobox_add 4 \
         --exhaustiveness 64 --num_modes 10 --cnn_scoring rescore \
         -o redocked.sdf

In [ ]:
# 5. Receptor-frame RMSD (the correct docking metric — do NOT realign).
# Both poses are already in the receptor frame; realigning (rdkit GetBestRMS/
# CalcRMS both do) masks bad poses and breaks the gate.
import json
from rdkit import Chem
import numpy as np

def canonical_heavy(mol):
    m = Chem.RemoveHs(mol)
    Chem.MolToSmiles(m, canonical=True)
    order = json.loads(m.GetProp('_smilesAtomOutputOrder'))
    return Chem.RenumberAtoms(m, list(order))

def receptor_frame_rmsd(crystal, pose):
    a, b = canonical_heavy(crystal), canonical_heavy(pose)
    assert a.GetNumAtoms() == b.GetNumAtoms()
    pa = a.GetConformer().GetPositions()
    pb = b.GetConformer().GetPositions()
    return float(np.sqrt(np.mean(np.sum((pa - pb) ** 2, axis=1))))

crystal = Chem.MolFromPDBFile('indirubin.pdb', removeHs=False, sanitize=False)
poses = [m for m in Chem.SDMolSupplier('redocked.sdf', removeHs=False, sanitize=False)]

for i, p in enumerate(poses[:5]):
    print(f"pose {i}: CNNscore={p.GetProp('CNNscore'):>6} "
          f"CNNaffinity={p.GetProp('CNNaffinity'):>8} "
          f"RMSD={receptor_frame_rmsd(crystal, p):5.2f} A")

best = min(range(len(poses)), key=lambda i: receptor_frame_rmsd(crystal, poses[i]))
r = receptor_frame_rmsd(crystal, poses[best])
print(f"\nbest pose RMSD = {r:.2f} A -> {'PASS' if r < 2.0 else 'FAIL'} (gate < 2.0 A)")

## If the gate passes → screening

Replace `ligand_prep.sdf` with a library. Good free sources:
- **DrugBank approved drugs** (~2,700) — for the repurposing play
- **ChEMBL AHR ligands** — the *J Med Chem* 2022 paper (10.1021/acs.jmedchem.2c00208) has AHR-active chemistries to seed from
- **REINVENT 4** (open source) for de novo design with docking in the scoring function

```bash
!./gnina -r receptor_prep.pdb -l library.sdf \
         --autobox_ligand indirubin.pdb --autobox_add 4 \
         --cnn fast --exhaustiveness 8 -o screen.sdf.gz
```

`--cnn fast` is the knowledge-distilled model (~16 s/compound on CPU) — the right
setting for high-throughput screens. Full results interpretation in
`research/TRIAGE_round1.md` and the project `STATUS.md`.